# 03 · Which k stations should Ulaanbaatar keep? (GA vs alternatives)

**Decision:** keep a subset S of the eligible stations. Dropped stations are estimated each day from the kept ones (ridge regression on log PM2.5).

**Fitness (lower is better):** `F(S) = mean wRMSE of dropped stations / σ + λ · MissRate`
- wRMSE: error in µg/m³, days above the threshold weigh 3×
- MissRate: share of exceedance days at dropped stations that the reduced network would predict as clean

**Protocol:** models fit on **train** (Oct–Dec 2025), subsets chosen on **validation** (Jan 2026).
The **test** months (Feb–Mar 2026) are scored once, in the last section, on 23 November.

Logic: `src/uaq/network.py`.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
cfg = load_config()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

In [ ]:
from uaq import audit_network as an, network
r = an.prepare(cfg)
fit = cfg["network"]["fitness"]
val_problem = network.Problem(r["wide"], r["masks"]["train"], r["masks"]["val"], r["threshold"], **fit)
print(f"{val_problem.n} stations, threshold {r['threshold']} µg/m³, σ = {val_problem.sigma:.1f}")

## 1 · What one fitness evaluation returns

In [ ]:
example = network.spatial_spread(val_problem, 4, r["lat"], r["lon"])
print("Subset:", val_problem.names(example))
val_problem.details(example)

## 2 · Every method at every k (validation month)
Random = median of 1,000 random subsets (the "no method" baseline); exhaustive = the true optimum; GA = best of 10 seeds.
This takes about 20 seconds.

In [ ]:
k_values = range(cfg["network"]["k_min"], val_problem.n)
res = network.compare(val_problem, k_values, cfg, r["lat"], r["lon"])
table = res.pivot(index="k", columns="method", values="F")
table["random (median)"] = res[res.method.str.startswith("random")].set_index("k")["random_median_F"]
table = table.drop(columns="random (best of draws)")
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
styles = {"exhaustive (optimum)": dict(color="k", lw=2.5, alpha=0.35), "genetic algorithm": dict(color="C3", marker="o"),
          "greedy forward": dict(color="C0", marker="s", ms=4), "highest mean": dict(color="C1", ls="--"),
          "spatial spread": dict(color="C2", ls="--"), "random (median)": dict(color="grey", ls=":")}
for m, st in styles.items():
    if m in table:
        ax.plot(table.index, table[m], label=m, **st)
ax.set_xlabel("stations kept (k)"); ax.set_ylabel("fitness F (lower is better)")
ax.set_title("Validation month: GA vs alternatives"); ax.legend(fontsize=8); plt.show()

## 3 · How close is each method to the optimum?

In [ ]:
opt = table["exhaustive (optimum)"]
gap = ((table.drop(columns="exhaustive (optimum)").sub(opt, axis=0)).div(opt, axis=0) * 100).round(1)
gap.columns = [c + " gap %" for c in gap.columns]
gap

In [ ]:
ga = res[res.method == "genetic algorithm"].set_index("k")
ga[["F", "ga_seed_F_mean", "ga_seed_F_worst", "seconds"]].round(4)

## 4 · GA convergence (one k)

In [ ]:
k = 5
fig, ax = plt.subplots(figsize=(7, 3.5))
for seed in cfg["network"]["ga"]["seeds"][:5]:
    _, _, hist = network.genetic_algorithm(val_problem, k, np.random.default_rng(seed),
                                           **{a: b for a, b in cfg["network"]["ga"].items() if a != "seeds"})
    ax.plot(hist, lw=1, label=f"seed {seed}")
ax.axhline(opt[k], color="k", ls="--", lw=1, label="optimum")
ax.set_xlabel("generation"); ax.set_ylabel("best F"); ax.set_title(f"GA convergence, k = {k}"); ax.legend(fontsize=8)
plt.show()

## 5 · Which stations does the GA keep?

In [ ]:
names = r["stations"].set_index("station_code")["name_mn"]
chosen = ga["stations"].apply(lambda s: ", ".join(f"{c} {names[c]}" for c in s))
chosen.to_frame("GA subset")

## 6 · Sensitivity to λ (weight on missed exceedances)
Does the recommended subset change if missed high-pollution days matter more or less?

In [ ]:
rows = []
for lam in (0.0, 0.5, 1.0):
    p = network.Problem(r["wide"], r["masks"]["train"], r["masks"]["val"], r["threshold"], **{**fit, "lambda_miss": lam})
    for k in (3, 5, 7):
        S = network.exhaustive(p, k)
        rows.append({"lambda": lam, "k": k, "stations": ", ".join(p.names(S)), **{a: round(b, 3) for a, b in p.details(S).items()}})
pd.DataFrame(rows)

## 7 · Save validation results

In [ ]:
out = ROOT / "outputs" / "results"; out.mkdir(parents=True, exist_ok=True)
res.to_csv(out / "network_validation.csv", index=False)
print("Saved", out / "network_validation.csv")

## 8 · FINAL TEST (run once, on 23 November)
Refit on train + validation, score the subsets each method chose on **validation** against the sealed Feb–Mar 2026 data.
Set `RUN_TEST = True` only once the forecasting model is also frozen. Do not change anything above after running this.

In [ ]:
RUN_TEST = False

if RUN_TEST:
    m = r["masks"]
    test_problem = network.Problem(r["wide"], m["train"] | m["val"], m["test"], r["threshold"], **fit)
    idx = {c: i for i, c in enumerate(test_problem.stations)}
    rows = []
    for _, row in res.iterrows():
        S = tuple(idx[c] for c in row["stations"])
        rows.append({"k": row["k"], "method": row["method"], **test_problem.details(S)})
    test_res = pd.DataFrame(rows)
    test_res.to_csv(out / "network_test.csv", index=False)
    display(test_res.pivot(index="k", columns="method", values="F").round(3))
else:
    print("Test set still sealed.")